# Assignment 3: Graph Visualization

This notebook completes the three required tasks:

1. **Load** a graph from a text file (Zachary's Karate Club — a classic social network from [SNAP](https://snap.stanford.edu/data/))
2. **Analyze** the graph, including **diameter** (computed by hand) and additional metrics
3. **Visualize** the graph with Matplotlib/NetworkX and export a file for **Gephi**

---

## 1. Install & Import Libraries

In [ ]:
# Uncomment if needed:
# !pip install networkx matplotlib

from collections import deque
from pathlib import Path

import matplotlib.pyplot as plt
import networkx as nx

print(f"NetworkX version: {nx.__version__}")

## 2. Load the Graph from a Text File

The dataset is stored as an **edge list** in `data/karate_club_edges.txt`. Each line contains two node IDs separated by whitespace. Comment lines (starting with `#`) are ignored.

Zachary collected this network in 1977 from a university karate club. After a dispute between two instructors, the club split into two factions — making this graph a classic benchmark for community detection.

In [ ]:
DATA_PATH = Path("data/karate_club_edges.txt")


def load_edge_list(path: Path) -> nx.Graph:
    """Read an undirected edge list from a text file."""
    G = nx.Graph()
    with path.open() as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            u, v = line.split()
            G.add_edge(int(u), int(v))
    return G


G = load_edge_list(DATA_PATH)

print(f"Loaded graph from: {DATA_PATH.resolve()}")
print(f"Nodes: {G.number_of_nodes()}")
print(f"Edges: {G.number_of_edges()}")
print(f"Sample edges: {list(G.edges())[:8]} ...")

## 3. Graph Analysis

### 3a. Diameter (hand-coded)

The **diameter** is the longest shortest path between any pair of nodes. For a connected graph:

$$\text{diameter}(G) = \max_{u,v} \, d(u, v)$$

We compute it with **breadth-first search (BFS)** from every node, then take the maximum distance found.

In [ ]:
def bfs_farthest_distance(G: nx.Graph, source) -> int:
    """Return the maximum shortest-path distance from source to any other node."""
    visited = {source: 0}
    queue = deque([source])

    while queue:
        node = queue.popleft()
        for neighbor in G.neighbors(node):
            if neighbor not in visited:
                visited[neighbor] = visited[node] + 1
                queue.append(neighbor)

    return max(visited.values())


def graph_diameter(G: nx.Graph) -> int:
    """Compute graph diameter for a connected undirected graph."""
    if not nx.is_connected(G):
        raise ValueError("Graph is disconnected; diameter is undefined for the full graph.")
    return max(bfs_farthest_distance(G, node) for node in G.nodes())


diameter_hand = graph_diameter(G)
diameter_nx = nx.diameter(G)

print(f"Diameter (hand-coded BFS): {diameter_hand}")
print(f"Diameter (NetworkX check): {diameter_nx}")

### 3b. Additional Metrics

Beyond diameter, we compute:
- **Average shortest path length** — typical separation between members
- **Average clustering coefficient** — how often friends of a member are also friends with each other
- **Graph density** — fraction of possible edges that actually exist
- **Degree centrality** — which members have the most direct ties

In [ ]:
avg_path_length = nx.average_shortest_path_length(G)
avg_clustering = nx.average_clustering(G)
density = nx.density(G)
degree_cent = nx.degree_centrality(G)

print("=== Graph Metrics ===")
print(f"Diameter                    : {diameter_hand}")
print(f"Average shortest path length: {avg_path_length:.4f}")
print(f"Average clustering coeff.   : {avg_clustering:.4f}")
print(f"Density                     : {density:.4f}")
print(f"Is connected                : {nx.is_connected(G)}")

print("\n=== Top 5 Nodes by Degree Centrality ===")
for node, score in sorted(degree_cent.items(), key=lambda x: -x[1])[:5]:
    print(f"  Node {node:>2}: {score:.4f} (degree {G.degree(node)})")

## 4. Visualization (Matplotlib + NetworkX)

We color nodes by **degree centrality** (darker = more connections) and size nodes by degree. This makes highly connected club members stand out visually.

In [ ]:
pos = nx.spring_layout(G, seed=42)
degrees = dict(G.degree())
node_sizes = [300 + 120 * degrees[n] for n in G.nodes()]
node_colors = [degree_cent[n] for n in G.nodes()]

plt.figure(figsize=(10, 8))
nodes = nx.draw_networkx_nodes(
    G, pos,
    node_size=node_sizes,
    node_color=node_colors,
    cmap=plt.cm.Blues,
    edgecolors="black",
    linewidths=0.8
)
nx.draw_networkx_edges(G, pos, alpha=0.35, width=1.2, edge_color="gray")
nx.draw_networkx_labels(G, pos, font_size=9, font_weight="bold")

plt.colorbar(nodes, label="Degree centrality", shrink=0.8)
plt.title(
    f"Zachary Karate Club Network\n"
    f"Diameter = {diameter_hand}, Avg path length = {avg_path_length:.2f}",
    fontsize=14,
    fontweight="bold"
)
plt.axis("off")
plt.tight_layout()
plt.savefig("karate_club_visualization.png", dpi=150, bbox_inches="tight")
plt.show()

print("Saved figure to karate_club_visualization.png")

## 5. Export for Gephi (or Neo4j)

To explore the graph in an external tool:

1. Open **Gephi** (https://gephi.org/)
2. Go to **File → Open** and select `data/karate_club.gexf`
3. Try layouts such as **ForceAtlas 2** or **Fruchterman Reingold**
4. Use **Statistics** to recompute metrics and **Appearance** to color by centrality

The GEXF format preserves node IDs and edge structure for interactive exploration.

In [ ]:
GEXF_PATH = Path("data/karate_club.gexf")
nx.write_gexf(G, GEXF_PATH)
print(f"Exported Gephi file: {GEXF_PATH.resolve()}")

## 6. Summary

| Task | What we did |
|------|-------------|
| **Load** | Parsed `data/karate_club_edges.txt` (78 edges, 34 nodes) |
| **Analyze** | Diameter via hand-coded BFS; avg path length, clustering, density, centrality |
| **Visualize** | Matplotlib plot saved as PNG; GEXF export for Gephi |

**Key findings for the Karate Club:**
- The network is small but well-connected (diameter = 5)
- Members are moderately clustered — friends of friends often know each other
- A few nodes (e.g., the club leaders) have much higher degree centrality than others

---
*Dataset: Zachary, W. W. (1977). An information flow model for conflict and fission in small groups. Journal of Anthropological Research, 33(4), 452–473.*